# 04 · Price Elasticity + Tariff Pricing Agent

Elasticity is estimated from UrbanEV dynamic-pricing grids (association, **not** causal). The decisive finding is that elasticity is **state-dependent**: demand is near-inelastic when stations are busy (ε≈0) and elastic when idle (ε≈−0.5). This is *why* dynamic pricing beats flat — surge the inelastic peaks, discount the elastic off-peak.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.elasticity import estimate_elasticity, estimate_elasticity_by_state
print('overall:', round(estimate_elasticity()['elasticity'],3))
state = estimate_elasticity_by_state(); state['by_state']

overall: -0.289


{'low': -0.5143301060478804,
 'mid': -0.12470445593510959,
 'high': -0.001,
 'vhigh': -0.001}

A single averaged elasticity hides this and (by Jensen's inequality) makes dynamic pricing *lose* to flat. State-dependent elasticity fixes it.

**Two operating points, reported honestly:**
- *Revenue-neutral* — same average tariff as the ₹15 flat rate; the gain is pure price discrimination.
- *Full learned* — also corrects the underpriced flat level (average price rises).

In [2]:
import pandas as pd
from src.data import load_unified
from src.agents.demand import DemandAgent
from src.agents.tariff import TariffAgent
from src.elasticity import make_eps_fn
panel = load_unified(); ag = DemandAgent(); ag.fit_eval(panel)
pred = ag.predictions[ag.predictions.source=='urbanev'].merge(
    panel[['timestamp','location_id','energy_kwh']], on=['timestamp','location_id'])
fn = make_eps_fn(state['by_state'])
neutral = TariffAgent(2.0, 1.5, eps_fn=fn, revenue_neutral=True)
full = TariffAgent(2.0, 1.5, eps_fn=fn)
_, kN = neutral.simulate(pred); _, kF = full.simulate(pred)
pd.DataFrame({'revenue_neutral':kN,'full_learned':kF}).loc[
    ['revenue_gain_pct','offpeak_uplift_pct','avg_price_multiplier']]

,revenue_neutral,full_learned
revenue_gain_pct,2.756690,20.486210
offpeak_uplift_pct,15.286617,6.074074
avg_price_multiplier,1.002721,1.202412
